# 📂🔍📂🔍 <span style="color: white; background-color: Purple"><b> Extração da Base de Headcount no Período </b></span></p>
      
🧩 <span style="color: MediumSlateBlue"><b> 1- Interação e Configuração do Período </b></span></p>
- Solicita ao usuário o Mês (ex: Janeiro, Fevereiro)
- Solicita o Ano (ex: 2026)
- Valida as entradas automaticamente, encerrando o programa caso o usuário digite um mês ou ano em formato inválido
- Isso garante flexibilidade para gerar relatórios retroativos ou atuais sob demanda

🧮 <span style="color: MediumSlateBlue"><b> 2- Cálculo Dinâmico da Janela de Tempo </b></span></p>
- O script utiliza a biblioteca calendar para calcular a lógica temporal
- Identifica qual é o último dia do mês escolhido (ex: 28, 30, 31)
- Cria a variável data_limite_fim, que representa o retrato exato do último dia daquele mês
- Essa data limite é o pilar para o cálculo de Headcount ativo

📥 <span style="color: MediumSlateBlue"><b> 3- Leitura da Base Mestre de RH </b></span></p>
- A automação acessa o diretório de rede: Controle_HC e Atestados.xlsx (aba "HC")
- Carrega a base completa de colaboradores
- Converter imediatamente as colunas data_admissao e data_rescisao para o formato datetime, garantindo que cálculos matemáticos de data funcionem perfeitamente

🎯 <span style="color: MediumSlateBlue"><b> 4- Lógica de Filtragem (Regra de Colaboradores Ativos) </b></span></p>
- O script aplica a regra para considerar um colaborador como ATIVO no mês selecionado:
    - Condição de Admissão: A data de admissão deve ser menor ou igual ao último dia do mês analisado
    - Condição de Rescisão: A data de rescisão deve ser nula (ainda trabalhando) OU maior que o último dia do mês analisado (foi demitido apenas em meses seguintes)
- O cruzamento dessas duas regras (condicao_admissao & condicao_rescisao) gera o dataframe exato de ativos

🧹 <span style="color: MediumSlateBlue"><b> 5- Tratamento e Padronização de Dados </b></span></p>
- Com os colaboradores filtrados, o pipeline executa correções de formatação
- Remove valores nulos
- Converte para número inteiro
- Transforma em texto (string) preenchendo com zeros à esquerda até ter 3 dígitos (ex: 1 vira 001, 25 vira 025)
- Converte a data_admissao do padrão de sistema (YYYY-MM-DD) para o padrão brasileiro (DD/MM/AAAA), facilitando a leitura no Excel final

📌 <span style="color: MediumSlateBlue"><b> 6- Seleção de Colunas Estratégicas </b></span></p>
- Para não gerar um arquivo pesado e com dados desnecessários, o script recorta a base mantendo apenas 13 colunas essenciais para o relatório de PCD e Aprendizes:
    - Registro e Nome
    - Sexo e Deficiente (Flag PCD)
    - Data de Admissão
    - Situação e Descrição da Rescisão
    - Cargo Completo e Centro de Custo
    - Salário Atual
    - Código da Empresa, Empresa e Unidade

📁 <span style="color: MediumSlateBlue"><b> 7- Exportação da Base Tratada </b></span></p>
- O arquivo final é salvo no diretório de BASES TRATADAS: BASE_RELATORIO_PCD_APRENDIZES.xlsx
- Aba nomeada como "Ativos"
- Sem índices numéricos do Pandas (arquivo limpo)
- O script verifica se a pasta de destino existe e, se não, a cria automaticamente (os.makedirs)

🧾 <span style="color: MediumSlateBlue"><b> 8- Tratamento de Erros e Resumo Final</b></span></p> 
- Ao final da execução, o sistema exibe no terminal:
    - Sucesso: A quantidade exata de colaboradores ativos exportados para aquele mês/ano
    - Tratamento de Erros: Caso o arquivo original não seja encontrado ou ocorra uma falha de memória/permissão, o script captura a exceção (try/except) e avisa o usuário de forma amigável, sem "quebrar" o terminal de forma abrupta

In [1]:
import os
import calendar
import threading
from datetime import datetime
import pandas as pd
import tkinter as tk
from tkinter import ttk, font as tkfont

# --- CONFIGURAÇÕES DE CAMINHOS ---
CAMINHO_ORIGEM = r"X:\Gestão de Pessoas\Analytics\10 - Relatórios\10.4 - HC e Atestados Médicos\Controle_HC e Atestados.xlsx"
PASTA_DESTINO = r"X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS"
ARQUIVO_DESTINO = os.path.join(PASTA_DESTINO, "BASE_RELATORIO_PCD_APRENDIZES.xlsx")

# Dicionário para converter o nome do mês em número
MESES = {
    "JANEIRO": 1, "FEVEREIRO": 2, "MARÇO": 3, "ABRIL": 4,
    "MAIO": 5, "JUNHO": 6, "JULHO": 7, "AGOSTO": 8,
    "SETEMBRO": 9, "OUTUBRO": 10, "NOVEMBRO": 11, "DEZEMBRO": 12
}

# --- PALETA DE CORES CORPORATIVAS ---
COR_BG               = "#F0F2F5"
COR_CARD             = "#FFFFFF"
COR_PRIMARIA         = "#1B3954"
COR_PRIMARIA_CLARA   = "#2C5F8A"
COR_TEXTO            = "#1A1A2E"
COR_TEXTO_CLARO      = "#555555"
COR_SUCESSO          = "#1E8449"
COR_SUCESSO_BG       = "#E8F6EF"
COR_ERRO             = "#C0392B"
COR_ERRO_BG          = "#FDEDEC"
COR_BORDA            = "#D5D8DC"
COR_BTN_TEXTO        = "#FFFFFF"
COR_BTN_HOVER        = "#142838"
COR_STATUS_BG        = "#F8F9FA"

# ====================================================================
#                        LÓGICA DE NEGÓCIO
# ====================================================================

def processar_dados(mes_nome, ano_str, callback_status):
    """
    Executa toda a pipeline de leitura, filtro, formatação e exportação.
    `callback_status` recebe tuplas (tipo, mensagem) para atualizar a interface.
    """
    mes_num = MESES[mes_nome]
    ano_num = int(ano_str)

    ultimo_dia = calendar.monthrange(ano_num, mes_num)[1]
    data_limite_fim = datetime(ano_num, mes_num, ultimo_dia)

    callback_status("info", f"Lendo o arquivo de origem...\n{os.path.basename(CAMINHO_ORIGEM)}")

    # --- LEITURA DOS DADOS ---
    df = pd.read_excel(CAMINHO_ORIGEM, sheet_name="HC")

    # Garante que as colunas de data estão no formato correto
    df['data_admissao'] = pd.to_datetime(df['data_admissao'], errors='coerce')
    df['data_rescisao'] = pd.to_datetime(df['data_rescisao'], errors='coerce')

    # --- LÓGICA DE FILTRO (ATIVOS NO MÊS) ---
    condicao_admissao = df['data_admissao'] <= data_limite_fim
    condicao_rescisao = df['data_rescisao'].isna() | (df['data_rescisao'] > data_limite_fim)
    df_ativos = df[condicao_admissao & condicao_rescisao].copy()

    # --- CORREÇÕES DE FORMATAÇÃO ---

    # 1. cod_empresa como texto com 3 dígitos
    if 'cod_empresa' in df_ativos.columns:
        df_ativos['cod_empresa'] = (
            df_ativos['cod_empresa']
            .fillna(0)
            .astype(int)
            .astype(str)
            .str.zfill(3)
        )

    # 2. data_admissao para DD/MM/AAAA
    if 'data_admissao' in df_ativos.columns:
        df_ativos['data_admissao'] = df_ativos['data_admissao'].dt.strftime('%d/%m/%Y')

    # --- SELEÇÃO DE COLUNAS ---
    colunas_desejadas = [
        "registro", "nome", "sexo", "deficiente", "data_admissao",
        "situacao", "descricao_rescisao", "cargo_completo",
        "centro_custo", "salario_atual", "cod_empresa", "empresa", "unidade"
    ]
    colunas_existentes = [col for col in colunas_desejadas if col in df_ativos.columns]
    df_resultado = df_ativos[colunas_existentes]

    # --- SALVAMENTO DO ARQUIVO ---
    callback_status("info", f"Salvando arquivo de destino...\n{os.path.basename(ARQUIVO_DESTINO)}")
    os.makedirs(PASTA_DESTINO, exist_ok=True)
    df_resultado.to_excel(ARQUIVO_DESTINO, index=False, sheet_name="Ativos")

    callback_status(
        "sucesso",
        f"Exportação concluída com sucesso!\n\n"
        f"Colaboradores ativos: {len(df_resultado)}\n"
        f"Período: {mes_nome}/{ano_num}\n"
        f"Arquivo: {ARQUIVO_DESTINO}"
    )

# ====================================================================
#                        INTERFACE TKINTER
# ====================================================================

class AppFiltroColaboradores:
    """Janela principal com interface Tkinter para filtro de colaboradores ativos."""

    LARGURA_JANELA = 620
    ALTURA_JANELA  = 660

    def __init__(self, root):
        self.root = root
        self._configurar_janela()
        self._configurar_estilos()
        self._construir_layout()

    # ----------------------------------------------------------------
    # Configuração da janela
    # ----------------------------------------------------------------
    def _configurar_janela(self):
        self.root.title("People Analytics — Filtro de Colaboradores Ativos")
        self.root.geometry(f"{self.LARGURA_JANELA}x{self.ALTURA_JANELA}")
        self.root.configure(bg=COR_BG)
        self.root.resizable(False, False)

        # Centralização
        self.root.update_idletasks()
        largura_tela  = self.root.winfo_screenwidth()
        altura_tela   = self.root.winfo_screenheight()
        x = (largura_tela - self.LARGURA_JANELA) // 2
        y = (altura_tela - self.ALTURA_JANELA) // 2
        self.root.geometry(f"+{x}+{y}")

    # ----------------------------------------------------------------
    # Estilos ttk
    # ----------------------------------------------------------------
    def _configurar_estilos(self):
        estilo = ttk.Style()
        estilo.theme_use("clam")

        fonte_titulo    = tkfont.Font(family="Segoe UI Semibold", size=18)
        fonte_subtitulo = tkfont.Font(family="Segoe UI", size=10)
        fonte_label     = tkfont.Font(family="Segoe UI Semibold", size=10)
        fonte_status    = tkfont.Font(family="Segoe UI", size=9)
        fonte_botao     = tkfont.Font(family="Segoe UI Semibold", size=11)

        # Label
        estilo.configure("TLabel", background=COR_CARD, foreground=COR_TEXTO, font=fonte_label)

        # Combobox
        estilo.configure("TCombobox",
                         fieldbackground=COR_CARD,
                         background=COR_CARD,
                         foreground=COR_TEXTO,
                         bordercolor=COR_BORDA,
                         lightcolor=COR_BORDA,
                         darkcolor=COR_BORDA,
                         arrowcolor=COR_PRIMARIA,
                         padding=6)
        estilo.map("TCombobox",
                   fieldbackground=[("readonly", COR_CARD)],
                   selectbackground=[("readonly", COR_PRIMARIA)],
                   selectforeground=[("readonly", COR_BTN_TEXTO)])

        # Entry
        estilo.configure("TEntry",
                         fieldbackground=COR_CARD,
                         foreground=COR_TEXTO,
                         bordercolor=COR_BORDA,
                         lightcolor=COR_BORDA,
                         darkcolor=COR_BORDA,
                         padding=6)

        # Botão
        estilo.configure("TButton",
                         background=COR_PRIMARIA,
                         foreground=COR_BTN_TEXTO,
                         font=fonte_botao,
                         borderwidth=0,
                         padding=(20, 12))
        estilo.map("TButton",
                   background=[("active", COR_BTN_HOVER), ("disabled", "#A0A0A0")],
                   foreground=[("active", COR_BTN_TEXTO)])

        # Frame
        estilo.configure("Card.TFrame", background=COR_CARD)
        estilo.configure("Status.TFrame", background=COR_STATUS_BG)
        estilo.configure("Bg.TFrame", background=COR_BG)

        # LabelFrame
        estilo.configure("TLabelframe",
                         background=COR_CARD,
                         foreground=COR_PRIMARIA,
                         bordercolor=COR_BORDA,
                         padding=(15, 10))
        estilo.configure("TLabelframe.Label",
                         background=COR_CARD,
                         foreground=COR_PRIMARIA,
                         font=fonte_label)

        # Guarda fontes para uso posterior
        self.fonte_titulo    = fonte_titulo
        self.fonte_subtitulo = fonte_subtitulo
        self.fonte_label     = fonte_label
        self.fonte_status    = fonte_status
        self.fonte_botao     = fonte_botao

    # ----------------------------------------------------------------
    # Construção do layout
    # ----------------------------------------------------------------
    def _construir_layout(self):

        # --- Container externo com margens ---
        frame_externo = ttk.Frame(self.root, style="Bg.TFrame")
        frame_externo.pack(fill="both", expand=True, padx=24, pady=20)

        # --- Card principal ---
        card = tk.Frame(frame_externo, bg=COR_CARD,
                        highlightbackground=COR_BORDA,
                        highlightthickness=1,
                        bd=0)
        card.pack(fill="both", expand=True)

        # Faixa de cabeçalho colorida
        faixa = tk.Frame(card, bg=COR_PRIMARIA, height=72)
        faixa.pack(fill="x")
        faixa.pack_propagate(False)

        lbl_titulo = tk.Label(faixa, text="Filtro de Colaboradores Ativos",
                              bg=COR_PRIMARIA, fg=COR_BTN_TEXTO,
                              font=self.fonte_titulo)
        lbl_titulo.pack(expand=True)

        # Subtítulo / descrição
        lbl_sub = tk.Label(card,
                           text="Selecione o mês e o ano de referência para filtrar\n"
                                "colaboradores ativos e exportar a base tratada em Excel.",
                           bg=COR_CARD, fg=COR_TEXTO_CLARO,
                           font=self.fonte_subtitulo, justify="center")
        lbl_sub.pack(pady=(18, 6))

        # Linha divisória sutil
        tk.Frame(card, bg=COR_BORDA, height=1).pack(fill="x", padx=30, pady=4)

        # --- Área de campos ---
        frame_campos = tk.Frame(card, bg=COR_CARD)
        frame_campos.pack(padx=40, pady=(16, 8), fill="x")

        # Mês
        lbl_mes = tk.Label(frame_campos, text="Mês de referência",
                          bg=COR_CARD, fg=COR_TEXTO, font=self.fonte_label)
        lbl_mes.grid(row=0, column=0, sticky="w", pady=(6, 2))

        lista_meses = list(MESES.keys())
        self.var_mes = tk.StringVar()
        self.cmb_mes = ttk.Combobox(frame_campos, textvariable=self.var_mes,
                                    values=lista_meses, state="readonly",
                                    font=self.fonte_label, width=28)
        self.cmb_mes.grid(row=1, column=0, sticky="ew", pady=(0, 14))
        self.cmb_mes.set(lista_meses[0])

        # Ano
        lbl_ano = tk.Label(frame_campos, text="Ano de referência",
                           bg=COR_CARD, fg=COR_TEXTO, font=self.fonte_label)
        lbl_ano.grid(row=2, column=0, sticky="w", pady=(6, 2))

        self.var_ano = tk.StringVar()
        self.ent_ano = ttk.Entry(frame_campos, textvariable=self.var_ano,
                                 font=self.fonte_label, width=30)
        self.ent_ano.grid(row=3, column=0, sticky="ew", pady=(0, 4))
        self.ent_ano.insert(0, str(datetime.now().year))

        # Dica de caminho de origem
        lbl_caminho = tk.Label(frame_campos,
                              text=f"Origem: {os.path.basename(CAMINHO_ORIGEM)}",
                              bg=COR_CARD, fg=COR_TEXTO_CLARO,
                              font=self.fonte_status, anchor="w")
        lbl_caminho.grid(row=4, column=0, sticky="w", pady=(10, 0))

        frame_campos.grid_columnconfigure(0, weight=1)

        # --- Botão de execução ---
        frame_btn = tk.Frame(card, bg=COR_CARD)
        frame_btn.pack(pady=(14, 8))

        self.btn_executar = tk.Button(
            frame_btn, text="▶  Processar e Exportar",
            bg=COR_PRIMARIA, fg=COR_BTN_TEXTO,
            activebackground=COR_BTN_HOVER, activeforeground=COR_BTN_TEXTO,
            font=self.fonte_botao, bd=0, padx=32, pady=10,
            cursor="hand2", command=self._on_processar
        )
        self.btn_executar.pack()

        # --- Área de status ---
        tk.Frame(card, bg=COR_BORDA, height=1).pack(fill="x", padx=30, pady=(4, 8))

        lbl_status_titulo = tk.Label(card, text="Status do processamento",
                                     bg=COR_CARD, fg=COR_TEXTO_CLARO,
                                     font=self.fonte_label, anchor="w")
        lbl_status_titulo.pack(fill="x", padx=40)

        self.frame_status = tk.Frame(card, bg=COR_STATUS_BG,
                                     highlightbackground=COR_BORDA,
                                     highlightthickness=1)
        self.frame_status.pack(fill="both", expand=True, padx=40, pady=(6, 20))

        self.txt_status = tk.Text(self.frame_status, bg=COR_STATUS_BG,
                                  fg=COR_TEXTO, font=self.fonte_status,
                                  bd=0, padx=12, pady=10, wrap="word",
                                  height=7, cursor="arrow")
        self.txt_status.pack(fill="both", expand=True)
        self._status_inicial()

    # ----------------------------------------------------------------
    # Status
    # ----------------------------------------------------------------
    def _status_inicial(self):
        self._set_status("info", "Aguardando seleção do período para iniciar o processamento.")

    def _set_status(self, tipo, mensagem):
        """Atualiza a área de status. Executa na main thread via after()."""
        self.txt_status.config(state="normal")
        self.txt_status.delete("1.0", "end")

        cor_bg = COR_STATUS_BG
        cor_fg = COR_TEXTO

        if tipo == "sucesso":
            cor_bg = COR_SUCESSO_BG
            cor_fg = COR_SUCESSO
        elif tipo == "erro":
            cor_bg = COR_ERRO_BG
            cor_fg = COR_ERRO

        self.txt_status.configure(bg=cor_bg, fg=cor_fg)
        self.frame_status.configure(bg=cor_bg)
        self.txt_status.insert("1.0", mensagem)
        self.txt_status.config(state="disabled")

    def _set_status_threadsafe(self, tipo, mensagem):
        """Wrapper thread-safe para atualizar o status a partir de threads secundárias."""
        self.root.after(0, lambda: self._set_status(tipo, mensagem))

    # ----------------------------------------------------------------
    # Validação
    # ----------------------------------------------------------------
    def _validar_entradas(self):
        mes = self.var_mes.get().strip().upper()
        ano = self.var_ano.get().strip()

        if mes not in MESES:
            return None, None, "Mês inválido. Selecione um mês da lista."

        if not ano.isdigit() or len(ano) != 4:
            return None, None, "Ano inválido. Digite um ano com 4 dígitos (ex: 2026)."

        ano_int = int(ano)
        if ano_int < 2000 or ano_int > 2100:
            return None, None, "Ano fora do intervalo permitido (2000–2100)."

        return mes, ano, None

    # ----------------------------------------------------------------
    # Ação do botão
    # ----------------------------------------------------------------
    def _on_processar(self):
        mes, ano, erro = self._validar_entradas()
        if erro:
            self._set_status("erro", erro)
            return

        # Desabilita botão e dá feedback visual
        self.btn_executar.config(state="disabled", text="⏳  Processando...")
        self._set_status("info", f"Iniciando processamento para {mes}/{ano}...")

        # Executa em thread separada para não congelar a interface
        thread = threading.Thread(
            target=self._executar_processamento,
            args=(mes, ano),
            daemon=True
        )
        thread.start()

    def _executar_processamento(self, mes, ano):
        """Roda a lógica de negócio em background e atualiza o status."""
        try:
            processar_dados(mes, ano, self._set_status_threadsafe)
        except FileNotFoundError:
            self._set_status_threadsafe(
                "erro",
                f"Arquivo não encontrado!\n\nCaminho:\n{CAMINHO_ORIGEM}"
            )
        except Exception as e:
            self._set_status_threadsafe("erro", f"Erro inesperado:\n{e}")
        finally:
            self.root.after(0, self._restaurar_botao)

    def _restaurar_botao(self):
        self.btn_executar.config(state="normal", text="▶  Processar e Exportar")

# ====================================================================
#                          PONTO DE ENTRADA
# ====================================================================

if __name__ == "__main__":
    root = tk.Tk()
    app = AppFiltroColaboradores(root)
    root.mainloop()